# Load data

In [170]:
import os

import numpy as np
from skimage.io import imread
import cv2 as cv
import numpy as np

import pandas as pd
from typing import List, Tuple, Callable, Optional
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [171]:
def load_data(base_dir, class_names, newSize):
    train_imgs = []
    labels = []

    for class_idx, class_name in enumerate(class_names):
        dir_name = base_dir + class_name
        for filename in os.listdir(dir_name):
            full_path = os.path.join(dir_name, filename)
            img = imread(full_path)#zwraca ndarry postaci xSize x ySize x colorDepth
            img = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
            img = cv.resize(img, newSize, interpolation=cv.INTER_AREA)# zwraca ndarray
            img = img / 255#normalizacja
            train_imgs.append(img)
            labels.append(class_idx)

    return np.array(train_imgs), np.array(labels)

In [172]:
def split_into_train_test(X, y, test_size=0.2, random_state=42):
    rand_indices = np.random.permutation(len(X))
    X = X[rand_indices]
    y = y[rand_indices]

    X = np.array([img.flatten() for img in X])
    return train_test_split(X, y, test_size=test_size, random_state=random_state)

In [173]:
base_dir = "../plantvillage dataset/grayscale/"

In [174]:
newSize = (64, 64)

# Binary classification model

In [175]:
def sigmoid(z):
    """Return the logistic function sigma(z) = 1/(1+exp(-z))."""
    return 1 / (1 + np.exp(-z))

def cost(Y, Yhat):
    """Return the cost function for predictions Yhat of classifications Y."""
    return (- Y @ np.log(Yhat.T) - (1 - Y) @ np.log(1 - Yhat.T)) / Y.shape[1]

def accuracy(Y, Yhat):
    """Return measure of the accuracy with which Yhat predicts Y."""
    return 1 - np.mean(np.abs(Y - Yhat.round()))

def model(X, w, b):
    """Apply the logistic model parameterized by w, b to features X."""
    z = w.T @ X + b
    return sigmoid(z)


def train(X, Y, alpha, max_it=1000, debug=False):
    """Train the logistic regression algorithm on the data X classified as Y."""

    nx = X.shape[0]
    m  = X.shape[1]
    print(f"m={m}, nx={nx}")

    w, b = np.random.random((nx,1)) * 0.01, 0.01
    # w, b = np.zeros((nx,1)), 0

    def propagate(w, b):
        """Propagate the training by advancing w, b to reduce the cost, J."""
        Yhat = model(X, w, b)
        w -= alpha / m * (X @ (Yhat - Y).T)
        b -= alpha / m * np.sum(Yhat - Y)
        J = np.squeeze(cost(Y, Yhat))

        if debug and not it % 100:
            # Provide an update on the progress we have made so far.
            print('{}: J = {}'.format(it, J))
            print('train accuracy = {:g}%'.format(accuracy(Y, Yhat) * 100))
        return w, b

    # Train the model by iteratively improving w, b.
    for it in range(max_it):
        w, b = propagate(w, b)
    return w, b


def predict(X, w, b):
    return model(X, w, b).round().flatten()

In [176]:
from sklearn.model_selection import KFold

def cross_validation_train(X, Y, alpha, max_it=1000, k=5, debug=False):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    accuracies = []
    best_accuracy = -np.inf
    best_w, best_b = None, None

    fold = 1
    for train_idx, val_idx in kf.split(X.T):
        if debug:
            print(f'Fold {fold}')

        X_train, X_val = X[:, train_idx], X[:, val_idx]
        Y_train, Y_val = Y[:, train_idx], Y[:, val_idx]

        w, b = train(X_train, Y_train, alpha, max_it, debug)

        Yhat_val = predict(X_val, w, b)
        acc = accuracy(Y_val.flatten(), Yhat_val)

        if debug:
            print(f'Validation accuracy for fold {fold}: {acc*100:.2f}%\n')

        accuracies.append(acc)

        if acc > best_accuracy:
            best_accuracy = acc
            best_w, best_b = w, b

        fold += 1

    if debug:
        mean_accuracy = np.mean(accuracies)
        print(f'Mean cross-validation accuracy: {mean_accuracy*100:.2f}%')
        print(f'Best accuracy: {best_accuracy*100:.2f}%')

    return best_w, best_b, accuracies

In [177]:
def eval(w, b, X_test, Y_expected):
    Y_predicted = predict(X_test, w, b)

    tp = 0
    tn = 0
    fp = 0
    fn = 0

    for i in range(len(Y_expected)):
        if Y_expected[i] == 1 and Y_predicted[i] == 1:
            tp += 1
        elif Y_expected[i] == 0 and Y_predicted[i] == 0:
            tn += 1
        elif Y_expected[i] == 0 and Y_predicted[i] == 1:
            fp += 1
        elif Y_expected[i] == 1 and Y_predicted[i] == 0:
            fn += 1

    accuracy = (tp + tn) / (tp + tn + fp + fn)
    print('Accuracy:', accuracy)

    precision = tp / (tp + fp)
    print('Precision:', precision)

    recall = tp / (tp + fn)
    print('Recall:', recall)

    fscore = (2 * precision * recall) / (precision + recall)
    print('F-score:', fscore)

# Blueberry___healthy vs Grape___healthy

In [178]:
X, y = load_data(base_dir, ["Blueberry___healthy", "Grape___healthy"], newSize)
X_train, X_test, y_train, y_test = split_into_train_test(X, y, test_size=0.2, random_state=42)

In [179]:
best_w, best_b, _ = cross_validation_train(
    X_train.T,
    y_train.reshape((1, y_train.shape[0])),
    alpha=0.01,
    max_it=5000,
    k=5,
    debug=True
)

Fold 1
m=1232, nx=4096
0: J = 8.907402803083306
train accuracy = 22.6461%
100: J = 0.4278626450667625
train accuracy = 76.8669%
200: J = 0.39055411623706304
train accuracy = 79.1396%
300: J = 0.37326317490435357
train accuracy = 80.6818%
400: J = 0.36169600715155087
train accuracy = 81.8182%
500: J = 0.35288505781861024
train accuracy = 83.1169%
600: J = 0.34570807182886876
train accuracy = 83.9286%
700: J = 0.339633753530533
train accuracy = 84.4156%
800: J = 0.3343648114957691
train accuracy = 85.1461%
900: J = 0.32971373207732213
train accuracy = 85.8766%
1000: J = 0.32555215510610175
train accuracy = 86.4448%
1100: J = 0.3217872602989118
train accuracy = 87.013%
1200: J = 0.3183493774620493
train accuracy = 87.0942%
1300: J = 0.31518481273891613
train accuracy = 87.4188%
1400: J = 0.3122513592433826
train accuracy = 87.4188%
1500: J = 0.30951531830810125
train accuracy = 87.5%
1600: J = 0.3069494358601178
train accuracy = 87.6623%
1700: J = 0.30453142615690315
train accuracy = 87.7

In [180]:
eval(best_w, best_b, X_test.T, y_test.T)

Accuracy: 0.8883116883116883
Precision: 0.734375
Recall: 0.6438356164383562
F-score: 0.6861313868613139


# Blueberry___healthy vs Strawberry___healthy

In [181]:
X, y = load_data(base_dir, ["Blueberry___healthy", "Strawberry___healthy"], newSize)
X_train, X_test, y_train, y_test = split_into_train_test(X, y, test_size=0.2, random_state=42)

In [182]:
best_w, best_b, _ = cross_validation_train(
    X_train.T,
    y_train.reshape((1, y_train.shape[0])),
    alpha=0.01,
    max_it=5000,
    k=5,
    debug=True
)

Fold 1
m=1252, nx=4096
0: J = 8.706860952796609
train accuracy = 22.4441%
100: J = 0.43837526469368693
train accuracy = 77.3962%
200: J = 0.40604420364973354
train accuracy = 78.9936%
300: J = 0.38662617949338113
train accuracy = 80.7508%
400: J = 0.3730821571733425
train accuracy = 81.869%
500: J = 0.36285732545770916
train accuracy = 82.4281%
600: J = 0.3547458169384693
train accuracy = 83.147%
700: J = 0.348077815841426
train accuracy = 83.9457%
800: J = 0.3424431006473965
train accuracy = 84.4249%
900: J = 0.33757439868854316
train accuracy = 84.9042%
1000: J = 0.33328987422054474
train accuracy = 85.3035%
1100: J = 0.32946164098461195
train accuracy = 85.7029%
1200: J = 0.32599727126693473
train accuracy = 85.623%
1300: J = 0.3228283884412665
train accuracy = 85.623%
1400: J = 0.31990336272395214
train accuracy = 85.8626%
1500: J = 0.31718249206271504
train accuracy = 86.262%
1600: J = 0.3146347412592713
train accuracy = 86.4217%
1700: J = 0.31223548639113663
train accuracy = 86.6

In [183]:
eval(best_w, best_b, X_test.T, y_test.T)

Accuracy: 0.8341836734693877
Precision: 0.6756756756756757
Recall: 0.5494505494505495
F-score: 0.6060606060606061


# Grape___healthy vs Strawberry___healthy

In [184]:
X, y = load_data(base_dir, ["Grape___healthy", "Strawberry___healthy"], newSize)
X_train, X_test, y_train, y_test = split_into_train_test(X, y, test_size=0.2, random_state=42)

In [185]:
best_w, best_b, _ = cross_validation_train(
    X_train.T,
    y_train.reshape((1, y_train.shape[0])),
    alpha=0.01,
    max_it=5000,
    k=5,
    debug=True
)

Fold 1
m=562, nx=4096
0: J = 5.292950758876363
train accuracy = 51.7794%
100: J = 1.0405920190935032
train accuracy = 48.3986%
200: J = 0.8363539774753787
train accuracy = 54.4484%
300: J = 0.6737664410137199
train accuracy = 61.9217%
400: J = 0.5613927701842389
train accuracy = 70.1068%
500: J = 0.48572300559970905
train accuracy = 76.1566%
600: J = 0.4347690258028126
train accuracy = 80.9609%
700: J = 0.40706776377121734
train accuracy = 83.9858%
800: J = 0.3953902257850843
train accuracy = 84.1637%
900: J = 0.38580119589211104
train accuracy = 84.8754%
1000: J = 0.3770588273907011
train accuracy = 84.8754%
1100: J = 0.3690224251593297
train accuracy = 85.5872%
1200: J = 0.36158772248375015
train accuracy = 86.121%
1300: J = 0.3546732216081863
train accuracy = 86.8327%
1400: J = 0.3482133376053897
train accuracy = 87.3665%
1500: J = 0.3421542079863538
train accuracy = 87.7224%
1600: J = 0.33645092362325824
train accuracy = 87.9004%
1700: J = 0.3310655955432751
train accuracy = 88.078

In [186]:
eval(best_w, best_b, X_test.T, y_test.T)

Accuracy: 0.7670454545454546
Precision: 0.7857142857142857
Recall: 0.7938144329896907
F-score: 0.7897435897435898
